<H1><b>Chapter 6. Document Loader</b></H1>
<hr/>

 <h3><b>1. Document의 구조</b></h3>

* Document 객체 -- LangChain의 기본 문서 객체이다.
  > page_content -- 문서의 실제 내용을 담고 있는 문자열이다.   
  > metadata -- 문서와 관련된 메타데이터를 저장하는 딕셔너리이다.
  
* RAG 구축을 위한 Office문서의 Markdown 변환 처리 도구의 이용이 효율적임. (by Microsoft 제공)
  > https://github.com/microsoft/markitdown

In [ ]:
from langchain_core.documents import Document

document = Document("Hello? This is a langchain's document!")

In [ ]:
# Docuemtn의 속성 조회
document.__dict__

In [ ]:
# Metadata 속성 추가
document.metadata["author"] = "Froggy"
document.metadata["total page"] = 1000

In [ ]:
# Metadata 속성 조회
document.metadata

<b>(1) Document Loader</b>

* 다양한 파일의 형식으로부터 불러온 내용을 문서(Document) 객체로 변환하는 역할을 합니다.
* 주요 Loader
  > PyPDFLoader: PDF 파일을 로드하는 로더입니다.   
  > CSVLoader: CSV 파일을 로드하는 로더입니다.   
  > UnstructuredHTMLLoader: HTML 파일을 로드하는 로더입니다.   
  > JSONLoader: JSON 파일을 로드하는 로더입니다.   
  > TextLoader: 텍스트 파일을 로드하는 로더입니다.   
  > DirectoryLoader: 디렉토리를 로드하는 로더입니다.   

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# 예제 파일 경로
FILE_PATH = "./data/SPRi AI Brief_11월호_산업동향_1105_F.pdf"

# 로더 설정
loader = PyPDFLoader(FILE_PATH)

<b>(2) load()</b>   

* 문서를 로드하여 반환합니다.   
* 반환된 결과는 List[Document] 형태입니다.   

In [ ]:
pip install pypdf

In [ ]:
# PDF 로더
docs = loader.load()

# 로드된 문서의 수 확인  --> PDF의 총 페이지수
len(docs)

In [ ]:
# 첫번째 문서 확인
docs[0]

<b>(3) load_and_split()</b>

* splitter 를 사용하여 문서를 분할하고 반환합니다.
* 반환된 결과는 List[Document] 형태입니다.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# 문열 분할기 설정
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=0)
# 문서 분할
docs = loader.load_and_split(text_splitter=text_splitter)

# 로드된 문서의 수 확인
print('len: ', len(docs))

# 첫번째 문서 확인
docs[0]

<b>(4) lazy_load()</b>

* generator 방식으로 문서를 로드합니다.

In [ ]:
# generator 방식으로 문서 로드
for doc in loader.lazy_load():
    print(doc.metadata)

<b>(5) aload()</b>

* 비동기(Async) 방식의 문서 로드

In [ ]:
# 문서를 async 방식으로 로드
adocs = loader.aload()

# 문서 로드
await adocs

<h3><b>2. PDF</b></h3>

In [ ]:
def show_metadata(docs):
    """ META 데이터의 조회 함수 """
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

<b>(1) PyPDF</b>

In [ ]:
pip install -qU pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# 예제 파일 경로
FILE_PATH = "./data/SPRi AI Brief_11월호_산업동향_1105_F.pdf"

# 파일 경로 설정
loader = PyPDFLoader(FILE_PATH)

# PDF 로더 초기화
docs = loader.load()

# 문서의 내용 출력
print(docs[10].page_content[:300])

In [ ]:
# 메타데이터 출력
show_metadata(docs)

<b>(2) PyPDF(OCR)</b>

* rapidocr-onnxruntime 패키지를 사용하여 이미지에서 텍스트를 추출

In [ ]:
pip install -qU rapidocr-onnxruntime

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# 예제 파일 경로
FILE_PATH = "./data/2103.15348v2.pdf"

# PDF 로더 초기화, 이미지 추출 옵션 활성화
#loader = PyPDFLoader("https://arxiv.org/pdf/2103.15348.pdf", extract_images=True)
loader = PyPDFLoader(FILE_PATH, extract_images=True)

# PDF 페이지 로드
docs = loader.load()

# 페이지 내용 접근
print(docs[4].page_content[:300])

In [ ]:
# 메타데이터 출력
show_metadata(docs)

<b>(3) PyMuPDF</b>

* PyMuPDF 는 속도 최적화가 되어 있으며, PDF 및 해당 페이지에 대한 자세한 메타데이터를 포함
* 페이지 당 하나의 문서를 반환

In [ ]:
pip install -qU pymupdf

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# 예제 파일 경로
FILE_PATH = "./data/SPRi AI Brief_11월호_산업동향_1105_F.pdf"

# PyMuPDF 로더 인스턴스 생성
loader = PyMuPDFLoader(FILE_PATH)

# 문서 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[10].page_content[:300])

In [ ]:
# 메타데이터 출력
show_metadata(docs)

<b>(4) PDFMiner</b>

In [ ]:
pip install pdfminer.six

In [ ]:
from langchain_community.document_loaders import PDFMinerLoader

# 예제 파일 경로
FILE_PATH = "./data/SPRi AI Brief_11월호_산업동향_1105_F.pdf"

# PDFMiner 로더 인스턴스 생성
loader = PDFMinerLoader(FILE_PATH)

# 데이터 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[0].page_content[:300])

In [ ]:
### HTML로 출력
from langchain_community.document_loaders import PDFMinerPDFasHTMLLoader

# 예제 파일 경로
FILE_PATH = "./data/SPRi AI Brief_11월호_산업동향_1105_F.pdf"

# PDFMinerPDFasHTMLLoader 인스턴스 생성
loader = PDFMinerPDFasHTMLLoader(FILE_PATH)

# 문서 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[0].page_content[:300])

In [ ]:
# 메타데이터 출력
show_metadata(docs)

<b>(5) PyPDF 디렉토리</b>

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# 디렉토리 경로
loader = PyPDFDirectoryLoader("data/")

# 문서 로드
docs = loader.load()

# 문서의 개수 출력
print(len(docs))

In [ ]:
# 문서의 내용 출력
print(docs[50].page_content[:300])

In [ ]:
# metadata 출력
print(docs[50].metadata)

<b>(6) PDFPlumber</b>

* PyMuPDF와 마찬가지로, 출력 문서는 PDF와 그 페이지에 대한 자세한 메타데이터를 포함하며, 페이지 당 하나의 문서를 반환

In [ ]:
pip install pdfplumber

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader

# 예제 파일 경로
FILE_PATH = "./data/SPRi AI Brief_11월호_산업동향_1105_F.pdf"

# PDF 문서 로더 인스턴스 생성
loader = PDFPlumberLoader(FILE_PATH)

# 문서 로딩
docs = loader.load()

# 첫 번째 문서 데이터 접근
print(docs[10].page_content[:300])

In [ ]:
# 메타데이터 출력
show_metadata(docs)

<h3><b>3. 한글(HWP)</b></h3>

* [한/글 문서 파일 형식: Python을 통한 HWP 포맷 파싱하기 (1)](https://tech.hancom.com/python-hwp-parsing-1/)
* [한/글 문서 파일 형식: Python을 통한 HWP 포맷 파싱하기 (2)](https://tech.hancom.com/python-hwp-parsing-2/)
* [한/글 문서 파일 형식: Python을 통한 HWPX 포맷 파싱하기 (1)](https://tech.hancom.com/python-hwpx-parsing-1/)
* [한/글 문서 파일 형식: Python을 통한 HWPX 포맷 파싱하기 (2)](https://tech.hancom.com/python-hwpx-parsing-2/)

<H3><b>4. CSV</b></H3>

* Comma-Separated Values (CSV) 파일은 쉼표로 값을 구분하는 구분된 텍스트 파일
* 파일의 각 줄은 데이터 레코드
* 각 레코드는 쉼표로 구분된 하나 이상의 필드로 구성

<b>(1) CSVLoader</b>

* CSV 데이터를 문서당 한 행씩 로드

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# CSV 로더 생성
loader = CSVLoader(file_path="./data/titanic.csv")

# 데이터 로드
docs = loader.load()

print(len(docs))
print(docs[0].metadata)

<b>(2) CSV 파싱 및 로딩 커스터마이징</b>

* csv_args에 대하여 <b>[csv module](https://docs.python.org/ko/3.14/library/csv.html)</b> 내용 참조
* source_column 인자를 사용하여 각 행에서 생성된 문서의 출처를 지정함. 그렇지 않으면 모든 문서의 출처로 file_path가 사용됨
  > ```python
loader = CSVLoader(
    file_path="./data/titanic.csv", source_column="PassengerId"
)  # CSV 로더 설정, 파일 경로 및 소스 컬럼 지정
  ```

In [ ]:
# 컬럼정보:
# PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked

# CSV 파일 경로
loader = CSVLoader(
    file_path="./data/titanic.csv",    
    csv_args={
        "delimiter": ",",  # 구분자
        "quotechar": '"',  # 인용 부호 문자
        "fieldnames": [
            "Passenger ID",
            "Survival (1: Survived, 0: Died)",
            "Passenger Class",
            "Name",
            "Sex",
            "Age",
            "Number of Siblings/Spouses Aboard",
            "Number of Parents/Children Aboard",
            "Ticket Number",
            "Fare",
            "Cabin",
            "Port of Embarkation",
        ],  # 필드 이름    
    },    
)

# 데이터 로드
docs = loader.load()

# 데이터 출력
print(docs[1].page_content)
print("-------------")
print(docs[1])

In [ ]:
# CSV 파일 경로
loader = CSVLoader(
    file_path="./data/titanic.csv",    
    source_column="PassengerId"       #출처 표시
)

# 데이터 로드
docs = loader.load()

# 데이터 출력
print(docs[1])

<b>(3) UnstructuredCSVLoader</b>

* UnstructuredCSVLoader를 사용하여 테이블을 로드할 수 있음
* UnstructuredCSVLoader를 사용하는 한 가지 장점은 "elements" 모드에서 사용할 경우, 메타데이터에서 테이블의 HTML 표현이 제공됨

In [ ]:
from langchain_community.document_loaders.csv_loader import UnstructuredCSVLoader

# 비구조화 CSV 로더 인스턴스 생성
loader = UnstructuredCSVLoader(file_path="./data/titanic.csv", mode="elements")

# 문서 로드
docs = loader.load()

# 첫 번째 문서의 HTML 텍스트 메타데이터 출력
print(docs[0].metadata["text_as_html"][:1000])
print("----------------")

from IPython.display import HTML, display
#display(HTML(docs[0].metadata["text_as_html"][:1000]))
display(HTML(docs[0].metadata["text_as_html"]))

<b>(4) DataFrameLoader</b>

* Pandas Library 사용

In [ ]:
import pandas as pd

# CSV 파일 읽기
df = pd.read_csv("./data/titanic.csv")

# 데이터프레임의 처음 다섯 행 조회
df.head()

In [ ]:
from langchain_community.document_loaders import DataFrameLoader

# 데이터 프레임 로더 설정, 페이지 내용 컬럼 지정
loader = DataFrameLoader(df, page_content_column="Name")

# 문서 로드
docs = loader.load()

# 데이터 출력
print("1) data ------------------>")
print(docs[0].page_content)

# 메타데이터 출력
print("\n" + "2) metadata ------------->")
print(docs[0].metadata)

In [ ]:
# 큰 테이블에 대한 지연 로딩, 전체 테이블을 메모리에 로드하지 않음
for row in loader.lazy_load():
    print("1) row --------------------->")
    print(row)
    print("\n" + "2) page_content ------------>")
    print(row.page_content)
    print("\n" + "3) metadata ---------------->")
    print(row.metadata)
    break  # 첫 행만 출력

<h3><b>5. Excel</b></h3>

<b>(1) Excel</b>

* .xlsx 및 .xls 파일 모두에서 작동
* "elements" 모드에서 로더를 사용하는 경우, 문서 메타데이터의 text_as_html 키 아래에서 Excel 파일의 HTML 표현이 제공됨

In [ ]:
pip install -qU langchain-community unstructured openpyxl

In [ ]:
pip install msoffcrypto-tool

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader

# UnstructuredExcelLoader 생성
loader = UnstructuredExcelLoader("./data/titanic.xlsx", mode="elements")

# 문서 로드
docs = loader.load()

# 문서 길이 출력
print(len(docs))

# 문서 출력
print(docs[0].page_content[:200])

In [ ]:
# metadata 의 text_as_html 출력
print(docs[0].metadata["text_as_html"][:1000])
print("----------------")

from IPython.display import HTML, display
#display(HTML(docs[0].metadata["text_as_html"][:1000]))
display(HTML(docs[0].metadata["text_as_html"]))

<b>(2) DataFrameLoader</b>

* Excel 파일을 로드하는 read_excel() 기능을 사용하여 DataFrame 으로 만든 뒤, 로드

In [ ]:
import pandas as pd

# Excel 파일 읽기
df = pd.read_excel("./data/titanic.xlsx")

In [ ]:
from langchain_community.document_loaders import DataFrameLoader

# 데이터 프레임 로더 설정, 페이지 내용 컬럼 지정
loader = DataFrameLoader(df, page_content_column="Name")

# 문서 로드
docs = loader.load()

# 데이터 출력
print("1) page_content ---------->")
print(docs[0].page_content)

# 메타데이터 출력
print("\n" + "2) metadata -------------->")
print(docs[0].metadata)

<h3><b>6. MS Word</b></h3>

<b>(1) Docx2txtLoader</b>

In [ ]:
pip install -qU docx2txt

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("./data/sample-word-document.docx")  # 문서 로더 초기화

docs = loader.load()  # 문서 로딩

print(len(docs))

print("1) page_content ---------->")
print(docs[0].page_content)
print("\n" + "2) metadata--------------->")
print(docs[0].metadata)

<b>(2) UnstructuredWordDocumentLoader</b>

In [ ]:
pip install -qU unstructured

In [ ]:
pip install --user -U nltk

In [ ]:
pip install certifi

In [ ]:
# NLTK 설치
# https://www.nltk.org/install.html

import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
    
nltk.download()

#팝업 윈도우에서 다운로드 처리

In [ ]:
# 1) 하나의 단일 Document로 load
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

# 비구조화된 워드 문서 로더 인스턴스화
loader = UnstructuredWordDocumentLoader("./data/sample-word-document.docx")

# 문서 로드
docs = loader.load()

print(len(docs))

print("1) page_content ---------->")
print(docs[0].page_content)
print("\n" + "2) metadata--------------->")
print(docs[0].metadata)

In [ ]:
# 2) element로 분할된 Document로 load
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

# 비구조화된 워드 문서 로더 인스턴스화
loader = UnstructuredWordDocumentLoader("./data/sample-word-document.docx", mode="elements")

# 문서 로드
docs = loader.load()

print(len(docs))

print("1) page_content ---------->")
print(docs[10].page_content)
print("\n" + "2) metadata--------------->")
print(docs[10].metadata)

<h3><b>7. MS Powerpoint</b></h3>

In [ ]:
pip install -qU unstructured python-pptx

In [ ]:
pip install -qU unstructured

In [ ]:
pip install --user -U nltk

In [ ]:
pip install certifi

In [ ]:
# NLTK 설치
# https://www.nltk.org/install.html

import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

#팝업 윈도우에서 다운로드 처리

In [ ]:
from langchain_community.document_loaders import UnstructuredPowerPointLoader

# UnstructuredPowerPointLoader 생성
loader = UnstructuredPowerPointLoader("./data/sample-ppt.pptx")

# 데이터 로드
docs = loader.load()

# 로드한 문서의 개수 출력
print(len(docs))

print("1) page_content ---------->")
print(docs[0].page_content)
print("\n" + "2) metadata--------------->")
print(docs[0].metadata)

<h3><b>8. 웹 문서(WebBaseLoader)</b></h3>

* [파이썬 BeautifulSoup 빠르게 효율적으로 설치하는 방법](https://apidog.com/kr/blog/how-to-install-beautifulsoup-on-python-quickly-and-efficiently-kr/)

<b>(1) WebBaseLoader</b>

* bs4 라이브러리를 사용하여 웹 페이지를 파싱
* Python의 웹 스크래핑 라이브러리인 Beautiful Soup 4 호출
```python
from bs4 import BeautifulSoup
import requests

# 웹 페이지 가져오기
url = 'https://example.com'
response = requests.get(url)
html = response.text

# BeautifulSoup 객체 생성
soup = BeautifulSoup(html, 'html.parser')

# 특정 태그에서 데이터 추출
title = soup.title.text
print("웹 페이지 제목:", title)

# 클래스 이름을 기반으로 요소 선택
articles = soup.find_all('div', class_='article')
for article in articles:
    print(article.text)

#1. find 
#첫 번째 매치되는 요소를 반환한다.
first_div = soup.find('div')

#2. find_all
#모든 요소를 반환한다/
all_divs = soup.find_all('div')

#3. select
#CSS 선택자를 사용하여 요소를 선택합니다
div_with_class = soup.select('.article')

#4. attrs
#요소의 속성에 접근한다.
div_class = soup.find('div')['class']

#5. parent
#부모 요소에 접근한다.
parent_div = soup.find('h1').parent

#6. find_next(), find_previous()
#다음 요소 혹은 이전 요소를 탐색한다.
next_paragraph = soup.find('p').find_next('p')

#7. find_all_next(), find_all_previous()
#다음 요소 혹은 이전 요소를 모두 탐색한다.
all_next_paragraphs = soup.find('p').find_all_next('p')

#8. strings
#요소 내의 모든 텍스트를 추출한다.
text_pieces = soup.find('div').strings
```


In [ ]:
# Python의 웹 스크래핑 라이브러리인 Beautiful Soup 4를 불러오는 코드
!pip install beautifulsoup4

# URL에서 HTML을 검색하는 데 일반적으로 사용
!pip install requests

In [ ]:
# HTTPS는 안됨
import bs4
from langchain_community.document_loaders import WebBaseLoader

# 뉴스기사 내용을 로드합니다.
loader = WebBaseLoader(
    web_paths=[
                "https://n.news.naver.com/article/437/0000378416",
                "https://n.news.naver.com/mnews/hotissue/article/092/0002340014?type=series&cid=2000063",
              ],
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
    header_template={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36",
    },
)

# ssl 인증 우회
loader.requests_kwargs = {"verify": False}

docs = loader.load()
print(f"문서의 수: {len(docs)}")

# 웹에서 가져온 결과를 출력
print(docs[0].page_content[:500])
print("===" * 10)
print(docs[1].page_content[:500])

In [ ]:
# HTTPS는 안됨
import requests

url = "http://quotes.toscrape.com" # 예제 웹사이트
#url = "https://apidog.com/kr/blog/how-to-install-beautifulsoup-on-python-quickly-and-efficiently-kr/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url, headers=headers, timeout=10, verify=False)
    response.raise_for_status() # HTTP 오류 확인
    html_content = response.content # 원시 바이트를 위해 .content 사용
except requests.exceptions.RequestException as e:
    print(f"{url} 가져오는 중 오류: {e}")
    html_content = None

html_content

<h3><b>9. 텍스트(TextLoader)</b></h3>

<b>(1) TXT Loader</b>

In [ ]:
from langchain_community.document_loaders import TextLoader

# File Path
FILE_PATH = "./data/appendix-keywords.txt"

# 텍스트 로더 생성
loader = TextLoader(FILE_PATH, encoding="UTF-8")  #encoding 처리하지 않으면 오류남

# 문서 로드
docs = loader.load()
print(f"문서의 수: {len(docs)}\n")
print("[메타데이터]\n")
print(docs[0].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[0].page_content[:500])

In [ ]:
# 단순 텍스트 파일 오픈
# File Path
FILE_PATH = "./data/appendix-keywords.txt"

f = open(FILE_PATH, encoding="UTF-8")

content = f.read()
f.close()
print(content)

<b>(2) TextLoader를 통한 파일 인코딩 자동 감지</b>

* silent_errors: DirectoryLoader에 silent_errors 매개변수를 전달하여 로드할 수 없는 파일을 건너뛰고 로드 프로세스를 계속할 수 있음
* autodetect_encoding: 또한 로더 클래스에 자동 감지_인코딩을 전달하여 실패하기 전에 파일 인코딩을 자동으로 감지하도록 요청할 수도 있음

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

path = "./data/"

text_loader_kwargs = {"autodetect_encoding": True}

loader = DirectoryLoader(
    path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    silent_errors=True,
    loader_kwargs=text_loader_kwargs,
)

docs = loader.load()

print(len(docs))

doc_sources = [doc.metadata["source"] for doc in docs]
doc_sources

print("[메타데이터]\n")
print(docs[1].metadata)
print("\n========= [앞부분] 미리보기 =========\n")
print(docs[1].page_content[:500])

<h3><b>10. JSON</b></h3>

<b>(1) JSON</b>

In [ ]:
# JSON 처리 도구 설치
!pip install jq

In [ ]:
import json
from pathlib import Path
from pprint import pprint


file_path = "./data/people.json"
data = json.loads(Path(file_path).read_text(encoding='utf-8'))  #encoding 지정하지 않으면 오류남

pprint(data)

type(data[0])

<b>(2) JSONLoader</b>

In [ ]:
from langchain_community.document_loaders import JSONLoader

# JSONLoader 생성
loader = JSONLoader(
    file_path="./data/people.json",
    jq_schema=".[].phoneNumbers",
    text_content=False,
)

# 문서 로드
docs = loader.load()

# 결과 출력
pprint(docs)

<h3><b>11. Markdown Loader</b></h3>

<b>(1) UnstructuredMarkdownLoader</b>

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# File Path 설정
FILE_PATH = "./data/Full-Markdown.md"

# Initialization
loader = UnstructuredMarkdownLoader(FILE_PATH)

# Laad
doc = loader.load()

# Document (metadata, page_content로 구성됨)
doc

<b>(2) Loader의 Mode 설정</b>

* "single": Returns the entire Markdown document as a single Document object.   
* "elements": Utilizes the unstructured library to split the document into elements like titles, narrative text, etc.,returning multiple Document objects, each representing an element. This mode is useful for preserving formatting and structure for tasks like RAG.

In [ ]:
# Example using "elements" mode
loader_elements = UnstructuredMarkdownLoader(FILE_PATH, mode="elements")

elements_documents = loader_elements.load()

print(len(elements_documents))

print("1) page_content ---------->")
print(elements_documents[15].page_content)
print("\n" + "2) metadata -------------->")
print(elements_documents[15].metadata)